# 📊 Interactive Stock Valuation & Risk Analyzer  
**Case Study: NVIDIA Corporation (NVDA)**

**Personal Project**  
AMA 106 – Introduction to Financial Economics  
Caraga State University | March 2026  
Prepared by: CJ John A. Paulin

---

**Purpose**:  
This interactive tool applies the core concepts from AMA 106 (Chapters 2–5) to a real stock.  
Users can change any assumption and instantly see the impact on valuation, risk, and investor decision-making.

In [14]:
# ====================== FINAL CLEAN & POLISHED VERSION (Warnings Suppressed) ======================
import warnings
warnings.filterwarnings("ignore")   # ← This removes the pink warnings

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from fpdf import FPDF

# ====================== WIDGETS ======================
ticker_widget = widgets.Text(value="NVDA", description="Ticker:")
shares_widget = widgets.FloatText(value=24.3, description="Shares (B):")
base_fcf_widget = widgets.FloatText(value=60.85, description="Base FCF 2025:")
growth1 = widgets.FloatSlider(value=30, min=0, max=50, description="2026 Growth %:")
growth2 = widgets.FloatSlider(value=30, min=0, max=50, description="2027 Growth %:")
growth3 = widgets.FloatSlider(value=25, min=0, max=50, description="2028 Growth %:")
growth4 = widgets.FloatSlider(value=20, min=0, max=50, description="2029 Growth %:")
growth5 = widgets.FloatSlider(value=15, min=0, max=50, description="2030 Growth %:")
rf_widget = widgets.FloatSlider(value=4.24, min=0, max=10, description="Risk-Free %:")
beta_widget = widgets.FloatSlider(value=2.31, min=0, max=5, description="Beta:")
p_boom_widget = widgets.FloatSlider(value=30, min=0, max=100, description="Boom %:")
p_normal_widget = widgets.FloatSlider(value=50, min=0, max=100, description="Normal %:")
r_boom_widget = widgets.FloatSlider(value=60, min=-100, max=200, description="Boom Return %:")
r_normal_widget = widgets.FloatSlider(value=20, min=-100, max=200, description="Normal Return %:")
r_recession_widget = widgets.FloatSlider(value=-40, min=-100, max=200, description="Recession Return %:")
A_widget = widgets.FloatSlider(value=4.0, min=0, max=10, description="Risk Aversion (A):")

calculate_button = widgets.Button(description="Calculate & Update Results", button_style='primary')
pdf_button = widgets.Button(description="📄 Download Client PDF Report", button_style='success')
output_area = widgets.Output()

last_results = {}

def on_calculate(b):
    global last_results
    with output_area:
        clear_output(wait=True)
        
        ticker = ticker_widget.value
        shares_outstanding = shares_widget.value
        base_fcf_2025 = base_fcf_widget.value
        growth_rates = [growth1.value/100, growth2.value/100, growth3.value/100, 
                        growth4.value/100, growth5.value/100]
        rf = rf_widget.value / 100
        beta = beta_widget.value
        p_boom = p_boom_widget.value / 100
        p_normal = p_normal_widget.value / 100
        p_recession = 1 - p_boom - p_normal
        r_boom = r_boom_widget.value / 100
        r_normal = r_normal_widget.value / 100
        r_recession = r_recession_widget.value / 100
        A = A_widget.value

        try:
            data = yf.download(ticker, period="2y", progress=False)
            current_price_live = float(data['Close'].iloc[-1].item())
        except:
            current_price_live = 180.00

        # DCF
        fcf = [base_fcf_2025]
        for g in growth_rates:
            fcf.append(fcf[-1] * (1 + g))
        fcf = fcf[1:]
        discount_factors = [(1 + rf) ** (i+1) for i in range(5)]
        pv_fcf = [fcf[i] / discount_factors[i] for i in range(5)]
        tv = fcf[-1] * (1 + 0.03) / (rf + 0.06*beta - 0.03)
        pv_tv = tv / discount_factors[-1]
        total_pv = sum(pv_fcf) + pv_tv
        intrinsic = float(total_pv / shares_outstanding)

        e_r = p_boom*r_boom + p_normal*r_normal + p_recession*r_recession
        var = (p_boom*(r_boom-e_r)**2 + p_normal*(r_normal-e_r)**2 + p_recession*(r_recession-e_r)**2)
        risk_premium = e_r - rf
        u_averse = e_r - 0.5 * A * var

        last_results = {
            "ticker": ticker,
            "current_price": current_price_live,
            "intrinsic": intrinsic,
            "e_r": e_r,
            "risk_premium": risk_premium,
            "u_averse": u_averse,
            "rf": rf
        }

        print(f"🔴 LIVE RESULTS for {ticker}")
        print(f"Current Price      : ${current_price_live:.2f}")
        print(f"Intrinsic Value    : ${intrinsic:.2f} → {'Undervalued' if intrinsic > current_price_live else 'Overvalued'}")
        print(f"Expected Return    : {e_r:.1%}")
        print(f"Risk Premium       : {risk_premium:.1%}")
        print(f"Risk-Averse Utility: {u_averse:.1%} (vs Risk-Free {rf:.1%})")

        # Charts
        states = ["Boom", "Normal", "Recession"]
        payoffs = [100000 * (1 + r) for r in [r_boom, r_normal, r_recession]]
        fig1 = go.Figure(data=[go.Bar(x=states, y=payoffs, marker_color=['green','blue','red'])])
        fig1.update_layout(title="Payoff Vector - ₱100,000 Investment", yaxis_title="Payoff (PHP)", height=400)
        fig1.show()

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(x=["Risk-Neutral", "Risk-Averse", "Risk-Free"], y=[e_r, u_averse, rf]))
        fig2.update_layout(title="Expected Utility Comparison", height=400)
        fig2.show()

def create_pdf(b):
    global last_results
    with output_area:
        if not last_results:
            print("❌ Please click 'Calculate & Update Results' first!")
            return
        try:
            pdf = FPDF()
            pdf.add_page()
            pdf.set_font("Arial", "B", 16)
            pdf.cell(0, 10, f"Stock Valuation Report - {last_results['ticker']}", ln=True, align="C")
            pdf.ln(15)
            pdf.set_font("Arial", size=12)
            pdf.cell(0, 10, f"Current Market Price : ${last_results['current_price']:.2f}", ln=True)
            pdf.cell(0, 10, f"Intrinsic Value      : ${last_results['intrinsic']:.2f}", ln=True)
            pdf.cell(0, 10, f"Valuation            : {'Undervalued' if last_results['intrinsic'] > last_results['current_price'] else 'Overvalued'}", ln=True)
            pdf.ln(10)
            pdf.cell(0, 10, f"Expected Return      : {last_results['e_r']:.1%}", ln=True)
            pdf.cell(0, 10, f"Risk Premium         : {last_results['risk_premium']:.1%}", ln=True)
            pdf.cell(0, 10, f"Risk-Averse Utility  : {last_results['u_averse']:.1%}", ln=True)
            pdf.cell(0, 10, f"Risk-Free Rate       : {last_results['rf']:.1%}", ln=True)
            
            filename = f"{last_results['ticker']}_Valuation_Report.pdf"
            pdf.output(filename)
            print(f"✅ PDF saved successfully → {filename}")
        except Exception as e:
            print(f"❌ PDF error: {e}")

calculate_button.on_click(on_calculate)
pdf_button.on_click(create_pdf)

# ====================== DISPLAY ======================
display(widgets.VBox([
    widgets.HTML("<h2>📊 Interactive Stock Valuation & Risk Analyzer</h2>"),
    widgets.HTML("<p><b>AMA 106 Personal Project • NVIDIA Case Study</b></p>"),
    widgets.HBox([ticker_widget, shares_widget]),
    widgets.HTML("<h3>DCF Inputs (Chapter 2 - Time Value of Money)</h3>"),
    base_fcf_widget, growth1, growth2, growth3, growth4, growth5,
    widgets.HTML("<h3>Risk Inputs (Chapter 5 - CAPM)</h3>"),
    rf_widget, beta_widget,
    widgets.HTML("<h3>States of Nature (Chapter 3)</h3>"),
    p_boom_widget, p_normal_widget,
    r_boom_widget, r_normal_widget, r_recession_widget,
    widgets.HTML("<h3>Utility Analysis (Chapter 3)</h3>"),
    A_widget,
    calculate_button,
    pdf_button,
    output_area
]))

## How to Use This Tool
- Adjust any slider or input
- Click **"Calculate & Update Results"** to see updated numbers and charts
- Click **"📄 Download Client PDF Report"** to get a clean report for clients

## AMA 106 Connection
This project directly applies:
- Chapter 2 → Discounting & DCF
- Chapter 3 → Expected Utility & States of Nature
- Chapter 4 → Risk & Return
- Chapter 5 → CAPM Beta & Risk Premium

**Ready for freelance / portfolio use.**